# Paso 1: Configuración del entorno y división estratificada

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, OrdinalEncoder, OneHotEncoder

# 1. Carga del dataset crudo
ruta_csv = 'clientes_banca_raw.csv'
df_raw = pd.read_csv(ruta_csv)

# 2. Separación de características predictoras (X) y variable objetivo (y)
X = df_raw.drop(columns=['abandono'])
y = df_raw['abandono']

# 3. Partición Train / Test estratificada (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Dimensiones de X_train inicial: {X_train.shape}")
print(f"Dimensiones de X_test inicial:  {X_test.shape}")

# Paso 2: Función para el tratamiento inicial de los datos

In [ ]:
def _descartar_columnas_no_utilizables(df: pd.DataFrame) -> pd.DataFrame:
    # Quitamos identificadores y columnas que dan pistas falsas o tienen demasiados nulos
    columnas_descarte = ['id_cliente', 'salario_mensual_aprox', 'comentarios_cliente', 'motivo_cancelacion']
    return df.drop(columns=columnas_descarte, errors='ignore')

def _sanitizar_centinelas_numericos(df: pd.DataFrame) -> pd.DataFrame:
    # Cambiamos números imposibles o códigos raros por valores vacíos
    if 'edad' in df.columns:
        df.loc[(df['edad'] <= 0) | (df['edad'] > 120), 'edad'] = np.nan
    if 'ingresos_anuales' in df.columns:
        df.loc[df['ingresos_anuales'] <= 0, 'ingresos_anuales'] = np.nan
    if 'saldo_cuenta' in df.columns:
        df['saldo_cuenta'] = df['saldo_cuenta'].replace(-999, np.nan)
    return df

def _sanitizar_categoricos(df: pd.DataFrame) -> pd.DataFrame:
    # Limpiamos espacios sobrantes, ponemos mayúsculas iniciales y quitamos signos de interrogación
    if 'tipo_cuenta' in df.columns:
        df['tipo_cuenta'] = df['tipo_cuenta'].astype(str).str.strip().str.capitalize()
        df['tipo_cuenta'] = df['tipo_cuenta'].replace({'Desconocido': np.nan, '?': np.nan, 'Nan': np.nan})

    if 'canal_preferido' in df.columns:
        df['canal_preferido'] = df['canal_preferido'].astype(str).str.strip()
        df['canal_preferido'] = df['canal_preferido'].replace({'???': np.nan, 'Nan': np.nan})

    if 'nivel_estudios' in df.columns:
        df['nivel_estudios'] = df['nivel_estudios'].astype(str).str.strip().replace({'Nan': np.nan, 'nan': np.nan})
    return df

def _feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    # Creamos las nuevas columnas que nos dan pistas útiles para el negocio
    if 'saldo_cuenta' in df.columns and 'ingresos_anuales' in df.columns:
        df['ratio_saldo_ingresos'] = df['saldo_cuenta'] / (df['ingresos_anuales'] + 1)

    if 'saldo_cuenta' in df.columns and 'num_productos' in df.columns:
        df['saldo_por_producto'] = df['saldo_cuenta'] / df['num_productos']

    if 'edad' in df.columns:
        limites_edad = [0, 30, 55, 120]
        etiquetas_edad = ['Joven', 'Adulto', 'Senior']
        df['grupo_edad'] = pd.cut(df['edad'], bins=limites_edad, labels=etiquetas_edad).astype(str)

    if 'antiguedad_meses' in df.columns:
        df['antiguedad_anios'] = df['antiguedad_meses'] / 12.0
        df = df.drop(columns=['antiguedad_meses'])
    return df

def preparar_datos_crudos(df_entrada: pd.DataFrame) -> pd.DataFrame:
    # Función principal que llama a las 4 subfunciones en orden
    df = df_entrada.copy()
    df = _descartar_columnas_no_utilizables(df)
    df = _sanitizar_centinelas_numericos(df)
    df = _sanitizar_categoricos(df)
    df = _feature_engineering(df)
    return df

Verificar columnas resultantes

In [ ]:
X_train_prep = preparar_datos_crudos(X_train)
X_test_prep = preparar_datos_crudos(X_test)

print(f"Columnas resultantes tras la preparación ({len(X_train_prep.columns)}):")
print(list(X_train_prep.columns))
X_train_prep.head(3)

## Ejercicio A

He dividido la función principal en cuatro partes para que el código quede mucho más ordenado y sea fácil de entender: una para quitar las columnas que no nos sirven, otra para cambiar números raros por nulos, otra para limpiar textos y una última para calcular las nuevas variables del negocio.

## Ejercicio B

In [ ]:
# 1. Comprobamos que se han eliminado las columnas descartadas
columnas_descarte = ['id_cliente', 'salario_mensual_aprox', 'comentarios_cliente', 'motivo_cancelacion']
print("¿Queda alguna columna descartada en X_train_prep?:")
print([col for col in columnas_descarte if col in X_train_prep.columns])

# 2. Comprobamos que se han añadido las columnas de feature engineering
columnas_nuevas = ['ratio_saldo_ingresos', 'saldo_por_producto', 'grupo_edad', 'antiguedad_anios']
print("\n¿Están todas las columnas nuevas en X_train_prep?:")
print([col for col in columnas_nuevas if col in X_train_prep.columns])

# 3. Valores únicos de tipo_cuenta y comprobación de nulos
print("\nValores únicos de tipo_cuenta:")
print(X_train_prep['tipo_cuenta'].unique())

print("\nNulos en tipo_cuenta:")
print(X_train_prep['tipo_cuenta'].isna().sum())

# Paso 3: Función de construcción del ColumnTransformer

In [ ]:
def construir_pipeline_preprocesamiento() -> ColumnTransformer:
    # 1. Columnas numéricas sin atípicos
    cols_num_standard = ['edad', 'num_productos', 'antiguedad_anios']

    # 2. Columnas numéricas con valores atípicos o asimétricas
    cols_num_robust = [
        'ingresos_anuales', 
        'saldo_cuenta', 
        'ratio_saldo_ingresos', 
        'saldo_por_producto'
    ]

    cols_ordinales = ['nivel_estudios', 'nivel_satisfaccion']
    cols_nominales = ['tipo_cuenta', 'canal_preferido', 'grupo_edad']
    cols_passthrough = ['tiene_tarjeta']

    # Sub-pipeline 1: Numérico Estándar (media + StandardScaler)
    pipeline_num_std = Pipeline([
        ('imputador', SimpleImputer(strategy='mean')),
        ('escalador', StandardScaler())
    ])

    # Sub-pipeline 2: Numérico Robusto (mediana + RobustScaler)
    pipeline_num_rob = Pipeline([
        ('imputador', SimpleImputer(strategy='median')),
        ('escalador', RobustScaler())
    ])

    # Sub-pipeline 3: Categórico Ordinal (moda + OrdinalEncoder + MinMaxScaler)
    orden_estudios = ['Primaria', 'Secundaria', 'Grado', 'Master', 'Doctorado']
    orden_satisfaccion = ['Muy Bajo', 'Bajo', 'Medio', 'Alto', 'Muy Alto']
    
    pipeline_ord = Pipeline([
        ('imputador', SimpleImputer(strategy='most_frequent')),
        ('codificador', OrdinalEncoder(categories=[orden_estudios, orden_satisfaccion])),
        ('escalador', MinMaxScaler())
    ])

    # Sub-pipeline 4: Categórico Nominal (moda + OneHotEncoder)
    pipeline_nom = Pipeline([
        ('imputador', SimpleImputer(strategy='most_frequent')),
        ('codificador', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
    ])

    # Ensamble de ramas en ColumnTransformer
    preprocesador = ColumnTransformer(
        transformers=[
            ('num_std', pipeline_num_std, cols_num_standard),
            ('num_rob', pipeline_num_rob, cols_num_robust),
            ('ord', pipeline_ord, cols_ordinales),
            ('nom', pipeline_nom, cols_nominales),
            ('bool', 'passthrough', cols_passthrough)
        ],
        remainder='drop'
    )

    # Configuramos la salida para que devuelva directamente DataFrames de Pandas
    preprocesador.set_output(transform='pandas')

    return preprocesador

Revisar arquitectura

In [ ]:
preprocesador = construir_pipeline_preprocesamiento()
preprocesador

## Ejercicio A

Sirve para que el sistema no se rompa si mañana un cliente nuevo escribe una opción que nunca habíamos visto en el entrenamiento. Si alguien pone un tipo de cuenta raro como Inversión y no tuviéramos ese parámetro, el programa daría un error fatal y se caería la web. Al poner ignore, simplemente le pone ceros en todas las columnas de cuenta y sigue funcionando sin fallos.

## Ejercicio B

La función devuelve el plano o la estructura del ColumnTransformer listo con todas sus ramas conectadas, pero todavía está completamente vacío. Aún no ha aprendido nada de nuestros datos porque no hemos ejecutado fit, así que todavía no sabe ni medias, ni modas ni categorías.

## Ejercicio C

Porque cada grupo de números se comporta de forma muy distinta. Variables como la edad o los años son normales y están equilibradas, así que StandardScaler funciona genial con ellas. En cambio, en el saldo o los sueldos hay clientes con muchísimo dinero que desviarían la media, por lo que usamos RobustScaler para que se fije en la mediana y no se deje engañar por esos extremos tan altos.

## Ejercicio D

Porque actúa como un escudo protector para cuando el sistema esté funcionando en la vida real. Aunque en nuestro archivo original no hubiera huecos vacíos, en la web un cliente puede olvidarse de rellenar una casilla o podemos generar un nulo nosotros al limpiar datos raros. Si no tuviéramos el imputador, al llegar un dato vacío el modelo se rompería al instante.

## Ejercicio E

Son columnas que no necesitan ningún tipo de transformación matemática ni escalado y que queremos dejar pasar tal cual a la salida final. El mejor ejemplo es tiene_tarjeta, porque ya es un valor binario con ceros y unos que el modelo entiende perfectamente sin tener que tocarlo.

# Paso 4: Función de ajuste y serialización

In [ ]:
def ajustar_y_guardar_preprocesador(
    preprocesador: ColumnTransformer, 
    X_train_prep: pd.DataFrame, 
    X_test_prep: pd.DataFrame, 
    ruta_guardado: str = 'preprocesador_banca.joblib'
):
    print("--> Ajustando preprocesador sobre X_train (fit_transform)...")
    X_train_final = preprocesador.fit_transform(X_train_prep)

    print("--> Aplicando preprocesador sobre X_test (transform)...")
    X_test_final = preprocesador.transform(X_test_prep)

    print(f"Dimensiones de X_train_final: {X_train_final.shape}")
    print(f"Dimensiones de X_test_final:  {X_test_final.shape}")

    # Guardamos el artefacto entrenado
    joblib.dump(preprocesador, ruta_guardado)
    print(f"Preprocesador exportado con éxito en: {ruta_guardado}")

    return X_train_final, X_test_final

In [ ]:
X_train_final, X_test_final = ajustar_y_guardar_preprocesador(
    preprocesador, X_train_prep, X_test_prep
)

X_train_final.head()

## Ejercicio A

La diferencia es que fit_transform primero aprende las reglas de los datos calculando las medias y modas y luego los transforma, mientras que transform solo aplica esas reglas ya aprendidas sin inventar nada nuevo. Jamás debemos hacer fit en test porque sería hacer trampa conociendo el futuro antes de tiempo y nuestro modelo parecería mejor de lo que realmente es en la vida real.

## Ejercicio B

In [ ]:
# Comprobamos que no queda ningún valor nulo en las matrices finales
print("Nulos totales en X_train_final:", X_train_final.isna().sum().sum())
print("Nulos totales en X_test_final:", X_test_final.isna().sum().sum())

# Paso 5: Simulación de inferencia en producción

In [ ]:
def transformar_cliente_nuevo(
    datos_cliente: dict, 
    ruta_artefacto: str = 'preprocesador_banca.joblib'
) -> pd.DataFrame:
    # 1. Cargamos el artefacto preprocesador entrenado
    preprocesador_cargado = joblib.load(ruta_artefacto)

    # 2. Convertimos a DataFrame de 1 fila
    df_nuevo = pd.DataFrame([datos_cliente])

    # 3. Aplicamos la misma limpieza previa determinista
    df_nuevo_prep = preparar_datos_crudos(df_nuevo)

    # 4. Transformamos con el pipeline sin hacer fit
    df_salida = preprocesador_cargado.transform(df_nuevo_prep)

    return df_salida

Cliente simulado

In [ ]:
cliente_simulado = {
    'id_cliente': 'CLI-9999',
    'edad': 42.0,
    'ingresos_anuales': 48000.0,
    'saldo_cuenta': 14200.0,
    'num_productos': 2,
    'tiene_tarjeta': 1,
    'antiguedad_meses': 60,
    'tipo_cuenta': '  ahorro  ',
    'canal_preferido': 'App Movil',
    'nivel_estudios': 'Grado',
    'nivel_satisfaccion': 'Alto'
}

df_cliente = transformar_cliente_nuevo(cliente_simulado)

print(f"Forma del DataFrame resultante: {df_cliente.shape}")
df_cliente

## Ejercicio A

El objetivo de esta función es simular exactamente lo que pasará en la aplicación real cuando un usuario rellene el formulario: coger sus datos en bruto tal cual los escribe, limpiarle los fallos y preparárselos en una sola fila lista para que el modelo le dé la predicción en milisegundos.

## Ejercicio B

Deberíamos usarla justo en el momento en que el servidor recibe la petición del cliente y antes de llamar al modelo. Primero recibimos el formulario con los datos en bruto, le pasamos esta función para convertirlo en los números exactos que espera el modelo, y con ese resultado ya podemos llamar a predict para responderle si el cliente se marcha o se queda.